In [ ]:
!pip install transformers torch accelerate
#Upgrade Jinja2
!pip install -U jinja2


In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print(f"Loading {MODEL_ID} on {device}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype).to(
    device
)
model.eval()

# Keep chat history across turns
chat_history = [{
    "role": "system",
    "content": (
        "You are a concise, helpful assistant for NIC e-Governance demos."
    ),
}]


def ask_bot(user_message: str, max_new_tokens: int = 150) -> str:
  chat_history.append({"role": "user", "content": user_message})

  prompt_text = tokenizer.apply_chat_template(
      chat_history, tokenize=False, add_generation_prompt=True
  )
  inputs = tokenizer([prompt_text], return_tensors="pt").to(device)

  with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,  # Deterministic output for reliable live demos
        pad_token_id=tokenizer.eos_token_id,
    )

  # Slice off the input prompt tokens so only the new reply is decoded
  generated_ids = output_ids[0][inputs["input_ids"].shape[1] :]
  reply = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

  chat_history.append({"role": "assistant", "content": reply})
  return reply


print("Chatbot ready!")

Loading Qwen/Qwen2.5-0.5B-Instruct on cpu...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Chatbot ready!


In [5]:
# 1. Quick scripted demo
print("User: Explain containerization in two short sentences.")
print("Bot :", ask_bot("Explain containerization in two short sentences."), "\n")

print("User: Give one example of how a district portal uses it.")
print("Bot :", ask_bot("Give one example of how a district portal uses it."))

User: Explain containerization in two short sentences.
Bot : Containerization is the process of packaging an application or software into a lightweight, self-contained executable that can run on any platform with a container runtime. This allows developers to deploy applications more efficiently and reduce the time required to build and test them. 

User: Give one example of how a district portal uses it.
Bot : A district portal might use containerization by deploying its web application as a Docker container, which can be easily scaled up or down based on demand. This allows the district to quickly deploy new features and updates without having to redeploy entire systems, reducing downtime and improving efficiency. Additionally, containerized applications can benefit from better performance and security due to their isolated environment.
